## Load environment

In [ ]:
import sys; sys.executable

'c:\\Users\\Usuario\\OneDrive - UNIVERSIDAD DE HUELVA\\Granada\\TrabajoFM\\scripts\\Python_Pipeline_SWAT_Pascal\\swat_pipeline\\trabajoFM\\.venv\\Scripts\\python.exe'

In [ ]:
from pathlib import Path
import sys
# Add project root to sys.path for imports
sys.path.insert(0, str(Path().resolve().parent))
from python_pipeline_scripts import utils, runner
from python_pipeline_scripts.provenance_report import write_provenance_reports

# Load config from config.yaml
config = utils.load_config(Path().resolve().parent / 'config' / 'config.yaml')

# Enable autoreload in Jupyter for live code updates
try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "1")
except NameError:
    pass  # Not in IPython/Jupyter


# this is were this notebook is located and from where relative paths will be resolved
import os
os.getcwd()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'c:\\Users\\Usuario\\OneDrive - UNIVERSIDAD DE HUELVA\\Granada\\TrabajoFM\\scripts\\Python_Pipeline_SWAT_Pascal\\swat_pipeline\\trabajoFM\\notebooks'

## test realization folder approach

In [ ]:

from python_pipeline_scripts.realizations import RealizationSpec, run_realizations_batch

In [ ]:
base = r"C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1" # must be the TxtInOut folder
realizations = [
RealizationSpec("diffuse_loads_1", r"C:\SWAT\RSWAT\cubillas\test\diffuse_loads_1"),
RealizationSpec("point_loads_1", r"C:\SWAT\RSWAT\cubillas\test\point_loads_1"),
]

patterns = [r"rcyr.*\.dat", r"fig.fig", r"0.*\.chm"]  # regex expressions as strings

outputs = ["output.std", "output.sub", "output.rch", "file.cio"]

In [ ]:
#res = run_realizations_batch(
#base,
#realizations,
#patterns,
#outputs,
#exe_path=r"C:\SWAT\ArcSWAT\swat_64rel.exe",
#results_root=r"C:/results/sim_runs",
#include_base_run=True,
#workspace_dir=r"C:/fast_disk/workspace" # optional, defaults to <base>_work/TxtInOut
#)

In [ ]:
from python_pipeline_scripts.rch_parser import load_multiple_rch_from_folders, load_output_rch



In [ ]:
results_mother_directory = r"C:\results\sim_runs"
output_folders = next(os.walk(results_mother_directory))[1]
output_paths = [os.path.join(results_mother_directory, folder) for folder in output_folders]
print(output_paths)

output_path_test = [r"C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_POINT_set-219\TxtInOut_1", r'C:\\results\\sim_runs\\out_point_loads_1']

['C:\\results\\sim_runs\\out_base_cubillas_BASE_set-219', 'C:\\results\\sim_runs\\out_diffuse_loads_1', 'C:\\results\\sim_runs\\out_point_loads_1']


In [ ]:
#rch_dfs = load_multiple_rch_from_folders(
#   output_path_test,
#    return_dict=True, name_from='parent', name_prefix='df_rch_'
#)

In [ ]:
#print(rch_dfs.keys())

### compare results

In [ ]:
from python_pipeline_scripts.comparisons import compare_dfs
#summary = compare_dfs(rch_dfs['df_rch_cubillas_BASE_POINT_set-219'], rch_dfs['df_rch_sim_runs'])

#print(summary)  

In [ ]:
# Jupyter single‑cell: CHM + Point pipelines in one Monte Carlo run with unified mean/extreme/random modes.
from pathlib import Path
import sys
from itertools import product
import random
import geopandas as gpd
import re
import time

# Make project root importable
sys.path.insert(0, str(Path().resolve().parent))

from python_pipeline_scripts import utils
from python_pipeline_scripts.raster_agg import raster_zonal_aggregation_to_gpkg
from python_pipeline_scripts.transforms.soil_chm import (
    read_n_p_means_from_csv_to_df,
    transform_apply_ops,
    transform_split_with_bounds,
    transform_write_chm_from_df,
)
from python_pipeline_scripts.transforms.point_dat import (
    read_population_by_subbasin_csv_to_df,
    transform_interpolate_years_wide,
    transform_build_point_load_timeseries,
    transform_write_point_dat_from_df,
)
from python_pipeline_scripts.mc_engine import run_monte_carlo
from python_pipeline_scripts.provenance_report import summarize_run, realization_report, build_upstream_inputs

DEBUG = True

def _resolve(p: str, base: Path) -> Path:
    pp = Path(p)
    return pp if pp.is_absolute() else (base / pp).resolve()



# ------------------------- 0) Resolve paths -------------------------
# CHM inputs
script_dir = Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\script DIFFUSE loads - input .chm")
output_gpkg = r"..\..\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.gpkg"
raster_folder = r"..\..\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster\temp_rasters"
zones_fp = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Archivos de Cesar Ruben Fernandez De Villaran San Juan - swat_cubillas\cubillas_hru\Watershed\Shapes\hru1.shp"
output_gpkg_p = _resolve(output_gpkg, script_dir)
raster_folder_p = _resolve(raster_folder, script_dir)
zones_fp_p = Path(zones_fp)
cfg = utils.load_config(Path("../config/config.yaml"))

# Aggregate CHM base data and export CSV
_ = raster_zonal_aggregation_to_gpkg(
    raster_folder=raster_folder_p,
    zones_fp=zones_fp_p,
    zone_field="HRU_GIS",
    label_field="OBJECTID",
    output_gpkg=output_gpkg_p,
    files_end_with="_rediam.tif",
    stat_operation="mean",
    raster_alias="full_name",
    zone_meaning="HRU",
    overwrite_cache=False,
    write_manifest=True,
    config=cfg,
)
layer_name = "values_by_hru"
gdf = gpd.read_file(output_gpkg_p, layer=layer_name)
chm_csv = str(output_gpkg_p).replace(".gpkg", ".csv")
gdf.drop(columns="geometry").to_csv(chm_csv, sep=";", index=False)
print(f"Exported CHM CSV: {chm_csv}")

# ------------------------- Point loads inputs -------------------------
# Folder where your point-load scripts live (for resolving relative paths)
point_script_dir = Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\script POINT loads - input .dat")

# Raster folder (HIPGDAC population rasters) and subbasins shapefile
pop_raster_folder = r"..\..\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\fgoerlich-HIPGDAC-ES-cd11f21\HIPGDAC-ES\1970-2021 copy"
subbasins_fp = r"..\..\Genil GEO_INFO_POOL\SWaT outputs\Cubillas\shapes cubillas\Sub_basin.shp"

# Output GPKG for population by subbasin (CSV will be written alongside)
pop_output_gpkg = r"..\..\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.gpkg"

pop_raster_folder_p = _resolve(pop_raster_folder, point_script_dir)
subbasins_fp_p = _resolve(subbasins_fp, point_script_dir)
pop_output_gpkg_p = _resolve(pop_output_gpkg, point_script_dir)
point_csv = Path(str(pop_output_gpkg_p).replace(".gpkg", ".csv"))

# Ensure population CSV exists; try to build with the inspiration function if missing
if not point_csv.exists():
    try:
        from python_pipeline_scripts.TEMP_point_dat_insparation import subbasinPopulationAggregationToGPKG
        print("Point CSV not found; building population GPKG/CSV...")
        _ = subbasinPopulationAggregationToGPKG(
            raster_folder=str(pop_raster_folder_p),
            sub_basin_fp=str(subbasins_fp_p),
            zone_field="GRIDCODE",
            output_gpkg=str(pop_output_gpkg_p),
            overwrite_cache=False
        )
        if not point_csv.exists():
            raise FileNotFoundError(f"Builder ran but CSV still missing: {point_csv}")
        print("Built population GPKG and CSV for point loads:", point_csv)
    except Exception as e:
        raise FileNotFoundError(
            f"Point loads CSV missing and could not be built automatically.\n"
            f"- Expected: {point_csv}\n"
            f"- Ensure HIPGDAC rasters and subbasins shapefile paths are correct,\n"
            f"  or run your population aggregation notebook to produce the CSV.\n"
            f"Cause: {e}"
        )

print("Point loads CSV:", point_csv)


# -------------------- 1) Monte Carlo spec --------------------
MC_SPEC = {
    "mode": "minmax",   # 'mean' | 'extreme' | 'random'
    "draws": 4,
    "transforms": [
       # CHM ops: compute base N,P (use factor for actual operation; keep lower/upper if you want bounds logged)
        {"type": "ops", "target": "chm", "name": "compute_base_ops", "ops": [
            {"src": "mean_Nitrogeno_total_porcent_resample_Rediam", "out": "N_total_mg_kg", "op": "mul",
            "factor": 10_000.0, "lower": None, "upper": None, "source": "deterministic"},
            {"src": "mean_Fosforo_mg_100g_P205_rediam", "out": "P_element_mg_kg", "op": "mul",
            "factor": 10.0 * 0.4364, "lower": None, "upper": None, "source": "deterministic"},
        ]},
        # CHM splits
        {"type": "split", "target": "chm", "src": "N_total_mg_kg", "renormalize": True,
         "outputs": [
             {"name": "Soil NO3 [mg/kg]", "mean": 0.02, "lower": 0.018, "upper": 0.022},
             {"name": "Soil organic N [mg/kg]", "mean": 0.98, "lower": 0.978, "upper": 0.982},
        ]},
        {"type": "ops", "target": "chm", "name": "derive_p_org_from_n", "ops": [
            {"src": "N_total_mg_kg", "out": "Soil organic P [mg/kg]", "op": "mul",
             "mean": 0.125, "lower": 0.1, "upper": 0.3},
        ]},
        {"type": "split", "target": "chm", "src": "P_element_mg_kg", "renormalize": True,
         "outputs": [
             {"name": "Soil labile P [mg/kg]", "mean": 1.0, "lower": 1.0, "upper": 1.0},
        ]},
        {"type": "write_chm", "target": "chm", "id_col": "HRU_GIS",
         "label_map": {
             "Soil NO3 [mg/kg]": "Soil NO3 [mg/kg]",
             "Soil organic N [mg/kg]": "Soil organic N [mg/kg]",
             "Soil labile P [mg/kg]": "Soil labile P [mg/kg]",
             "Soil organic P [mg/kg]": "Soil organic P [mg/kg]",
         }},
        # Point: interpolate (optional)
        {"type": "interpolate_years_wide", "target": "point", "id_col": "GRIDCODE", "year_start": 1970, "year_end": 2021},
        # Point: build timeseries from population + mg/L specs
        {"type": "build_point", "target": "point", "id_col": "GRIDCODE", "wastewater_lppd": 150.0,
         "mgL_values": {
             "ORGNYR": {"mean": 15, "lower": 12, "upper": 18},
             "ORGPYR": {"mean": 3,  "lower": 2.5, "upper": 3.5},
             "NO3YR":  {"mean": 0,  "lower": 0,   "upper": 0},
             "NH3YR":  {"mean": 25, "lower": 20,  "upper": 30},
             "NO2YR":  {"mean": 0,  "lower": 0,   "upper": 0},
             "MINPYR": {"mean": 5,  "lower": 4,   "upper": 6},
             "SEDYR":  {"mean": 720,"lower": 600, "upper": 800},
             "CBODYR": {"mean": 220,"lower": 180, "upper": 260},
             "DISOXYR":{"mean": 2.5,"lower": 2.0, "upper": 3.0},
             "CHLAYR": {"mean": 0.001,"lower": 0.0,"upper": 0.002}
         },
         "out_columns": ["YEAR","FLOYR","SEDYR","ORGNYR","ORGPYR","NO3YR","NH3YR","NO2YR","MINPYR","CBODYR","DISOXYR","CHLAYR"]},
        # Point: write rcyr_*.dat
        {"type": "write_point_dat", "target": "point", "id_col": "GRIDCODE",
         "columns_order": ["YEAR","FLOYR","SEDYR","ORGNYR","ORGPYR","NO3YR","NH3YR","NO2YR","MINPYR","CBODYR","DISOXYR","CHLAYR"],
         "start_year": 1970, "end_year": 2021},
    ]
}

# ------------------ 2) Build transforms & defaults ------------------
base_txtinout = r"C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1"
realizations_root = r"C:\SWAT\RSWAT\cubillas\mc_realizations"
results_root = r"C:\SWAT\RSWAT\cubillas\mc_results"
base_txtinout_p = Path(base_txtinout)
realizations_root_p = Path(realizations_root)
results_root_p = Path(results_root)

# Upstream inputs (CHM + Point)
manifest_file = Path(str(output_gpkg_p) + ".manifest.json")
upstream_chm = build_upstream_inputs(
    raster_folder=raster_folder_p,
    pattern="*_rediam.tif",
    zones_fp=zones_fp_p,
    gpkg_path=output_gpkg_p,
    csv_path=Path(chm_csv),
)
upstream_point = [point_csv]
upstream = list({Path(p) for p in (upstream_chm + upstream_point)})

# Wrapper to run CHM/Point transforms on their respective data within a single container
def wrap_on(key, inner):
    def fn(data, dest_dir, rng, params, rp):
        df = data[key]
        out_df, paths = inner(df, dest_dir, rng, params, rp)
        data[key] = out_df
        return data, paths
    return fn


# Helper: ensure each op has the operative parameter set for transform_apply_ops
def _coerce_ops_for_apply(ops_list):
    out = []
    for spec in ops_list:
        spec = dict(spec)
        op = spec.get("op") or spec.get("mode")
        # If the operative parameter is missing, fall back to 'standard' value if present.
        # We accept keys 'factor' (mul), 'delta' (add), 'value' (set).
        if op in ("mul", "relative"):
            if "factor" not in spec and "standard" in spec:
                spec["factor"] = float(spec["standard"])
            if "factor" not in spec and "mean" in spec:
                spec["factor"] = float(spec["mean"])
        elif op in ("add", "absolute"):
            if "delta" not in spec and "standard" in spec:
                spec["delta"] = float(spec["standard"])
            if "delta" not in spec and "mean" in spec:
                spec["delta"] = float(spec["mean"])
        elif op == "set":
            if "value" not in spec and "standard" in spec:
                spec["value"] = float(spec["standard"])
            if "value" not in spec and "mean" in spec:
                spec["value"] = float(spec["mean"])
        out.append(spec)
    return out



# Map MC_SPEC types to wrapped transforms
# This must be the first transform executed.
def transform_init_copy(data, dest_dir, rng, params, rp):
    out = {}
    for k, v in data.items():
        try:
            out[k] = v.copy(deep=True)  # pandas DataFrame
        except Exception:
            import copy as _copy
            out[k] = _copy.deepcopy(v)
    return out, []

# Make these your first entries so each realization starts from a clean copy
transforms = [transform_init_copy]
transforms_params = [{}]

for t in MC_SPEC["transforms"]:
    tgt = t.get("target", "chm")  # 'chm' or 'point'

# For ops mapping: use the coerce helper so transform_apply_ops always has a factor/delta/value set
    if t["type"] == "ops":
        transforms.append(wrap_on(tgt, transform_apply_ops))
        coerced_ops = _coerce_ops_for_apply(t["ops"])  # ensure operative param is present
        transforms_params.append({
            "ops": coerced_ops,
            "input_source": str(manifest_file) if manifest_file.exists() else None,
            "debug": DEBUG,
        })

    elif t["type"] == "split":
        transforms.append(wrap_on(tgt, transform_split_with_bounds))
        outputs = [{"name": o["name"], "mean": o["mean"], "lower": o["lower"], "upper": o["upper"], "source": MC_SPEC["mode"]} for o in t["outputs"]]
        transforms_params.append({
            "src": t["src"],
            "renormalize": bool(t.get("renormalize", True)),
            "outputs": outputs,
            "input_source": str(manifest_file) if manifest_file.exists() else None,
            "debug": DEBUG,
        })
    elif t["type"] == "write_chm":
        transforms.append(wrap_on(tgt, transform_write_chm_from_df))
        transforms_params.append({
            "id_col": t["id_col"],
            "label_map": t["label_map"],
        })
    elif t["type"] == "interpolate_years_wide":
        transforms.append(wrap_on("point", transform_interpolate_years_wide))
        transforms_params.append({
            "id_col": t.get("id_col", "GRIDCODE"),
            "year_start": int(t["year_start"]),
            "year_end": int(t["year_end"]),
            "keep_existing": bool(t.get("keep_existing", True)),
        })
    elif t["type"] == "build_point":
        transforms.append(wrap_on("point", transform_build_point_load_timeseries))
        transforms_params.append({
            "id_col": t.get("id_col", "GRIDCODE"),
            "wastewater_lppd": float(t.get("wastewater_lppd", 150.0)),
            "mgL_values": t["mgL_values"],
            "out_columns": t.get("out_columns"),
            "round_to": int(t.get("round_to", 6)),
        })
    elif t["type"] == "write_point_dat":
        transforms.append(wrap_on("point", transform_write_point_dat_from_df))
        transforms_params.append({
            "id_col": t.get("id_col", "GRIDCODE"),
            "columns_order": t.get("columns_order"),
            "start_year": t.get("start_year"),
            "end_year": t.get("end_year"),
        })
    else:
        raise ValueError(f"Unknown transform type: {t['type']}")

# ------------------ 3) Build per‑realization overrides (mean/extreme/random) ------------------
def _op_param_name(kind: str) -> str:
    return {"mul":"factor","relative":"factor","add":"delta","absolute":"delta","set":"value"}.get(kind, "factor")

def _has_bounds(lo, hi) -> bool:
    try: return lo is not None and hi is not None and float(hi) != float(lo)
    except: return False

def build_extreme_overrides(spec):
    from itertools import product
    per_tf_opts = []
    for t in spec["transforms"]:
        tt = t["type"]

        if tt == "ops":
            bundles, any_b = [], False
            for op in t["ops"]:
                kind = op.get("op","mul"); key = _op_param_name(kind)
                lo, hi = op.get("lower"), op.get("upper")
                if _has_bounds(lo, hi):
                    any_b = True
                    bundles.append([{**op, key: float(lo), "source":"extreme_lower"},
                                    {**op, key: float(hi), "source":"extreme_upper"}])
                else:
                    bundles.append([{**op}])
            per_tf_opts.append([{"ops":[dict(b) for b in combo]} for combo in product(*bundles)] if any_b else [{}])

        elif tt == "split":
            outs = []
            for o in t["outputs"]:
                lo, hi = float(o["lower"]), float(o["upper"])
                name = o["name"]
                if abs(hi-lo) < 1e-12:
                    outs.append([{"name": name, "ratio": lo, "source":"lower_equals_upper"}])
                else:
                    outs.append([{"name": name, "ratio": lo, "source":"extreme_lower"},
                                 {"name": name, "ratio": hi, "source":"extreme_upper"}])
            per_tf_opts.append([{"outputs": list(combo)} for combo in product(*outs)])

        elif tt == "build_point":
            mg_sets = []
            for var, meta in t["mgL_values"].items():
                lo, hi = meta.get("lower"), meta.get("upper")
                if _has_bounds(lo, hi):
                    mg_sets.append([{var: {**meta, "mgL": float(lo), "source":"extreme_lower"}},
                                    {var: {**meta, "mgL": float(hi), "source":"extreme_upper"}}])
                else:
                    mg_sets.append([{var: {**meta, "mgL": float(meta.get("standard", meta.get("mean", meta.get("mgL", 0.0)))), "source":"standard"}}])
            combos = []
            for combo in product(*mg_sets):
                merged = {}
                for d in combo: merged.update(d)
                combos.append({"mgL_values": merged})
            per_tf_opts.append(combos)

        else:
            per_tf_opts.append([{}])
    return [list(c) for c in product(*per_tf_opts)]

def build_random_overrides(spec, n_draws, seed=None):
    import random; rnd = random.Random(seed)
    draws = []
    for _ in range(n_draws):
        per_tf = []
        for t in spec["transforms"]:
            tt = t["type"]

            if tt == "ops":
                bundle = []
                for op in t["ops"]:
                    kind = op.get("op","mul"); key = _op_param_name(kind)
                    lo, hi = op.get("lower"), op.get("upper")
                    if _has_bounds(lo, hi):
                        bundle.append({**op, key: rnd.uniform(float(lo), float(hi)), "source":"random"})
                    else:
                        bundle.append(dict(op))
                per_tf.append({"ops": bundle})

            elif tt == "split":
                outs = []
                for o in t["outputs"]:
                    lo, hi = float(o["lower"]), float(o["upper"])
                    if abs(hi-lo) < 1e-12:
                        outs.append({"name": o["name"], "ratio": lo, "source":"lower_equals_upper"})
                    else:
                        outs.append({"name": o["name"], "ratio": rnd.uniform(lo, hi), "source":"random"})
                per_tf.append({"outputs": outs})

            elif tt == "build_point":
                mg = {}
                for var, meta in t["mgL_values"].items():
                    lo, hi = meta.get("lower"), meta.get("upper")
                    if _has_bounds(lo, hi):
                        mg[var] = {**meta, "mgL": rnd.uniform(float(lo), float(hi)), "source":"random"}
                    else:
                        mg[var] = {**meta, "mgL": float(meta.get("standard", meta.get("mean", meta.get("mgL", 0.0)))), "source":"standard"}
                per_tf.append({"mgL_values": mg})

            else:
                per_tf.append({})
        draws.append(per_tf)
    return draws

def build_mean_overrides(spec):
    per_tf = []
    for t in spec["transforms"]:
        tt = t["type"]

        if tt == "ops":
            bundle = []
            for op in t["ops"]:
                kind = op.get("op","mul"); key = _op_param_name(kind)
                std = op.get("standard", op.get("mean", op.get(key)))
                bundle.append({**op, key: float(std), "source":"standard"})
            per_tf.append({"ops": bundle})

        elif tt == "split":
            outs = [{"name": o["name"], "ratio": float(o.get("standard", o["mean"])), "source":"standard"} for o in t["outputs"]]
            per_tf.append({"outputs": outs})

        elif tt == "build_point":
            mg = {}
            for var, meta in t["mgL_values"].items():
                std = float(meta.get("standard", meta.get("mean", meta.get("mgL", 0.0))))
                mg[var] = {**meta, "mgL": std, "source": "standard"}
            per_tf.append({"mgL_values": mg})

        else:
            per_tf.append({})
    return [per_tf]


def build_lower_upper_overrides(spec):
    """
    Returns two realizations of per-transform overrides:
      - Realization 0: all bounded params set to their lower bound (source='all_lower')
      - Realization 1: all bounded params set to their upper bound (source='all_upper')
    Unbounded transforms remain unchanged across both runs.
    """
    lowers = []
    uppers = []

    for t in spec["transforms"]:
        tt = t["type"]

        if tt == "ops":
            low_ops, up_ops = [], []
            for op in t["ops"]:
                op = dict(op)
                kind = op.get("op","mul")
                key = _op_param_name(kind)
                lo, hi = op.get("lower"), op.get("upper")

                # Lower
                if _has_bounds(lo, hi):
                    op_low = {**op, key: float(lo), "source": "all_lower"}
                else:
                    op_low = dict(op)
                low_ops.append(op_low)

                # Upper
                if _has_bounds(lo, hi):
                    op_up = {**op, key: float(hi), "source": "all_upper"}
                else:
                    op_up = dict(op)
                up_ops.append(op_up)

            lowers.append({"ops": low_ops})
            uppers.append({"ops": up_ops})

        elif tt == "split":
            low_outs, up_outs = [], []
            for o in t.get("outputs", []):
                o = dict(o)
                lo, hi = o.get("lower"), o.get("upper")
                name = o.get("name")
                o_low = {"name": name}
                o_up  = {"name": name}
                if _has_bounds(lo, hi):
                    o_low.update({"ratio": float(lo), "source": "all_lower"})
                    o_up.update({"ratio": float(hi), "source": "all_upper"})
                low_outs.append(o_low)
                up_outs.append(o_up)
            lowers.append({"outputs": low_outs})
            uppers.append({"outputs": up_outs})

        elif tt == "build_point":
            low_map, up_map = {}, {}
            for var, meta in t.get("mgL_values", {}).items():
                meta = dict(meta)
                lo, hi = meta.get("lower"), meta.get("upper")
                if _has_bounds(lo, hi):
                    low_map[var] = {**meta, "mgL": float(lo), "source": "all_lower"}
                    up_map[var]  = {**meta, "mgL": float(hi), "source": "all_upper"}
                else:
                    std = meta.get("mgL", meta.get("standard", meta.get("mean", 0.0)))
                    low_map[var] = {**meta, "mgL": float(std), "source": meta.get("source", "standard")}
                    up_map[var]  = {**meta, "mgL": float(std), "source": meta.get("source", "standard")}
            lowers.append({"mgL_values": low_map})
            uppers.append({"mgL_values": up_map})

        else:
            # write_chm, write_point_dat, interpolate_years_wide, etc.
            lowers.append({})
            uppers.append({})

    return [lowers, uppers]

# === Select mode and build per_realization_params ===
MODE = MC_SPEC["mode"].strip().lower()

if MODE in ("minmax", "lower_upper"):
    per_realization_params = build_lower_upper_overrides(MC_SPEC)
elif MODE == "extreme":
    per_realization_params = build_extreme_overrides(MC_SPEC)
elif MODE == "random":
    import time
    OVERRIDE_SEED = int(time.time() * 1000) & 0xFFFFFFFF  # vary across runs
    per_realization_params = build_random_overrides(MC_SPEC, MC_SPEC.get("draws", 1), seed=OVERRIDE_SEED)
elif MODE == "mean":
    per_realization_params = build_mean_overrides(MC_SPEC)
else:
    raise ValueError(f"Unknown mode: {MODE}")

# === IMPORTANT: align overrides to transforms if you have an initial copy transform at index 0 ===
if isinstance(per_realization_params, list) and per_realization_params:
    if isinstance(per_realization_params[0], list):
        per_realization_params = [[{}] + step_list for step_list in per_realization_params]
    elif isinstance(per_realization_params[0], dict):
        per_realization_params = [[{}] + [per_realization_params[0].get(t.__name__, {}) for t in transforms[1:]]]

# Optional: quick diagnostic to confirm alignment for the first realization
if DEBUG:
    print("[diag] transforms:", [t.__name__ for t in transforms])
    print("[diag] overrides[0] lens:", len(per_realization_params[0]), "==", len(transforms))
    for i, (tfn, ovar) in enumerate(zip(transforms, per_realization_params[0])):
        if i > 0 and (ovar.get("ops") or ovar.get("outputs") or ovar.get("mgL_values")):
            print(f"[diag] t[{i}] {tfn.__name__} override keys:", list(ovar.keys()))


# ------------------ 4) Run MC (write CHMs + DATs only) ------------------
RUN_MODEL = True

# Aggregator returns a container with both CHM and Point dataframes
aggregator = lambda: {
    "chm": read_n_p_means_from_csv_to_df(
        chm_csv,
        id_col="HRU_GIS",
        n_col="mean_Nitrogeno_total_porcent_resample_Rediam",
        p_col="mean_Fosforo_mg_100g_P205_rediam",
    ),
    "point": read_population_by_subbasin_csv_to_df(point_csv, id_col="GRIDCODE", sep=";"),
}


results = run_monte_carlo(
    N=len(per_realization_params),
    base_txtinout=base_txtinout_p,
    realization_root=realizations_root_p,
    results_root=results_root_p,
    # Link CHMs and point rcyr DAT files
    link_file_regexes=[r"^[0-9]+\.chm$", r"^rcyr_.*\.dat$"],
    outputs_to_copy=["output.std", "*.rch"],
    aggregator=aggregator,
    transforms=transforms,
    transforms_params=transforms_params,
    per_realization_params=per_realization_params,
    exe_path=None,
    seed=0,
    expect_plus=False,
    config=cfg,
    include_base_run=False,
    create_workspace_copy=True,
    force_recreate_workspace=True,
    report=True,
    run_model=RUN_MODEL,
    upstream_inputs=upstream,
    manifest_file=manifest_file if manifest_file.exists() else None,
    auto_attach_manifest=True,
)

ok = sum(1 for r in results if r.success)
run_id = results[0].run_id if results else -1
print(f"Created {len(results)} realizations; {ok} succeeded. run_id={run_id}")
for r in results:
    print(f"- {r.name}: id={r.realization_id} run_id={r.run_id} success={r.success} folder={r.folder}")

# ------------------ 5) Summaries ------------------
print("\n=== Monte Carlo run summary ===")
print(summarize_run(run_id))

if results:
    one_id = results[0].realization_id
    print(f"\n=== Single realization report (id={one_id}) ===")
    print(realization_report(one_id))


2025-09-03 17:18:22,304 | INFO | python_pipeline_scripts.raster_agg | Zonal aggregation start | zones=C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Archivos de Cesar Ruben Fernandez De Villaran San Juan - swat_cubillas\cubillas_hru\Watershed\Shapes\hru1.shp | rasters_dir=C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster\temp_rasters
2025-09-03 17:18:23,053 | INFO | python_pipeline_scripts.raster_agg | Valid HRU features: 878 | bounds=[ 438912.5        4123462.50012207  470812.5        4158362.50012207]
2025-09-03 17:18:23,058 | INFO | python_pipeline_scripts.raster_agg | Found 2 raster(s) matching '_rediam.tif'
2025-09-03 17:18:23,069 | INFO | python_pipeline_scripts.raster_agg | Reprojected (cached) -> C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster\temp_rasters\temp_cache\Fosforo_mg_100g_P205_

Exported CHM CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.csv
Point loads CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.csv
[diag] transforms: ['transform_init_copy', 'fn', 'fn', 'fn', 'fn', 'fn', 'fn', 'fn', 'fn']
[diag] overrides[0] lens: 9 == 9
[diag] t[1] fn override keys: ['ops']
[diag] t[2] fn override keys: ['outputs']
[diag] t[3] fn override keys: ['ops']
[diag] t[4] fn override keys: ['outputs']
[diag] t[7] fn override keys: ['mgL_values']


2025-09-03 17:20:54,211 | INFO | python_pipeline_scripts.transforms.point_dat | Loaded subbasin CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Point loads\Population data\HIPGDAC-ES-v1.0.0\custom_aggregations\open_source\cubillas_population.csv | rows=17
2025-09-03 17:20:54,533 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_Nitrogeno_total_porcent_resample_Rediam -> N_total_mg_kg | value=10000.0 | mean=None lower=None upper=None
2025-09-03 17:20:54,542 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_Fosforo_mg_100g_P205_rediam -> P_element_mg_kg | value=4.364 | mean=None lower=None upper=None
2025-09-03 17:20:54,548 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] split N_total_mg_kg -> Soil NO3 [mg/kg] | ratio=0.018072 (mean=0.02 lower=0.018 upper=0.022) renorm=True
2025-09-03 17:20:54,567 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] split N

[point] normalized years found: [1970, 1981, 1991, 2001, 2011, 2021]


2025-09-03 17:21:02,032 | INFO | python_pipeline_scripts.transforms.point_dat | Wrote 17 rcyr_*.dat files to C:\SWAT\RSWAT\cubillas\mc_realizations\run000061_real000288_1
2025-09-03 17:21:03,870 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_Nitrogeno_total_porcent_resample_Rediam -> N_total_mg_kg | value=10000.0 | mean=None lower=None upper=None
2025-09-03 17:21:03,876 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] op mul: mean_Fosforo_mg_100g_P205_rediam -> P_element_mg_kg | value=4.364 | mean=None lower=None upper=None
2025-09-03 17:21:03,880 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] split N_total_mg_kg -> Soil NO3 [mg/kg] | ratio=0.021912 (mean=0.02 lower=0.018 upper=0.022) renorm=True
2025-09-03 17:21:03,882 | INFO | python_pipeline_scripts.transforms.soil_chm | [debug] split N_total_mg_kg -> Soil organic N [mg/kg] | ratio=0.978088 (mean=0.98 lower=0.978 upper=0.982) renorm=True
2025-09-03 17:21:03,887 | INFO | pyth

[point] normalized years found: [1970, 1981, 1991, 2001, 2011, 2021]


2025-09-03 17:21:10,116 | INFO | python_pipeline_scripts.transforms.point_dat | Wrote 17 rcyr_*.dat files to C:\SWAT\RSWAT\cubillas\mc_realizations\run000061_real000289_2
2025-09-03 17:21:12,119 | INFO | python_pipeline_scripts.realizations | Removing existing workspace to recreate: C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1_work\TxtInOut
2025-09-03 17:21:18,440 | INFO | python_pipeline_scripts.realizations | Creating full workspace copy at C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1_work\TxtInOut
2025-09-03 17:22:12,564 | INFO | python_pipeline_scripts.realizations | === Realization 'Soil labile P [mg/kg]' start | folder=C:\SWAT\RSWAT\cubillas\mc_realizations\run000061_real000288_1 ===
2025-09-03 17:22:12,599 | INFO | python_pipeline_scripts.realizations | Linked (hardlink): 000010001.chm -> C:\SWAT\RSWAT\cubillas\mc_realizations\run000061_real000288_1\000010001.chm
2025-09-03 17:22:12,614 | INFO | python_pipeline_

Created 2 realizations; 2 succeeded. run_id=61
- Soil labile P [mg/kg]: id=289 run_id=61 success=True folder=C:\SWAT\RSWAT\cubillas\mc_realizations\run000061_real000289_2
- Soil labile P [mg/kg]: id=289 run_id=61 success=True folder=C:\SWAT\RSWAT\cubillas\mc_realizations\run000061_real000289_2

=== Monte Carlo run summary ===
Run 61: realizations=1 ids=[289]
Time span: 2025-09-03T15:21:03.831380+00:00 → 2025-09-03T15:21:03.831380+00:00
Names:
  - run000061_real000289_2
Transforms:
  - transform_init_copy
  - fn
  - ops_choices
  - split_choices
  - transform_interpolate_years_wide
  - point_mgL_choices
  - transform_write_point_dat_from_df
Inputs (union):
  - C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Archivos de Cesar Ruben Fernandez De Villaran San Juan - swat_cubillas\cubillas_hru\Watershed\Shapes\hru1.shp
  - C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster\temp_rasters\Fosforo_mg_

In [ ]:
from python_pipeline_scripts.provenance_report import summarize_run, realization_report, build_upstream_inputs

print(realization_report(one_id+1))
#print(realization_report(266))

Realization id=287 run_id=60 name=run000060_real000287_2
Created: 2025-09-03T14:36:37.308365+00:00
Engine: module=python_pipeline_scripts.mc_engine version=0.1.0 seed=0 N=2
Base TxtInOut: C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1
Realization folder: C:\SWAT\RSWAT\cubillas\mc_realizations\run000060_real000287_2

Summary (transform | in -> out | bounds | value | source | renorm):
  - op_mul: mean_Nitrogeno_total_porcent_resample_Rediam -> N_total_mg_kg | - | 10000.000000 | deterministic | 
  - op_mul: mean_Fosforo_mg_100g_P205_rediam -> P_element_mg_kg | - | 4.364000 | deterministic | 
  - split_with_bounds: N_total_mg_kg -> Soil NO3 [mg/kg] | [0.018..0.022] | 0.021912 | all_upper | renormalized
  - split_with_bounds: N_total_mg_kg -> Soil organic N [mg/kg] | [0.978..0.982] | 0.978088 | all_upper | renormalized
  - op_mul: N_total_mg_kg -> Soil organic P [mg/kg] | [0.1..0.3] | 0.300000 | all_upper | 
  - split_with_bounds: P_element_mg_kg -> Soil labi

In [ ]:
from python_pipeline_scripts.provenance_report import read_ledger, _ledger_path_default
p = _ledger_path_default()
recs = read_ledger()
print("Ledger:", p)
print("Total records:", len(recs))
print("Run records:", len([r for r in recs if r.get('run_id') == run_id or r.get('engine', {}).get('run_id') == run_id]))
print("IDs in run:", [r.get('id') for r in recs if r.get('run_id') == run_id or r.get('engine', {}).get('run_id') == run_id])

Ledger: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\provenance\realizations.jsonl
Total records: 166
Run records: 4
IDs in run: [266, 267, 268, 269]


In [ ]:
from python_pipeline_scripts.provenance_report import _ledger_path_default, read_ledger

DEFAULT_LEDGER = _ledger_path_default()
print("Default ledger:", DEFAULT_LEDGER)

recs = read_ledger(DEFAULT_LEDGER)
print("Default ledger records:", len(recs))

# Show a quick summary of IDs and run_ids present
ids = [r.get("id") for r in recs if r.get("id") is not None]
runs = [r.get("run_id") or (r.get("engine") or {}).get("run_id") for r in recs]
print("ID range:", (min(ids) if ids else None, max(ids) if ids else None))
print("Last 20 run_ids (unique):", sorted({x for x in runs if x is not None})[-20:])

# Optionally show the last few records’ ids and names
for r in recs[-5:]:
    print("id:", r.get("id"), "run_id:", r.get("run_id") or (r.get("engine") or {}).get("run_id"), "name:", r.get("name"))


Default ledger: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\provenance\realizations.jsonl
Default ledger records: 162
ID range: (5, 239)
Run_ids (unique): [20, 21, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 37, 38, 39, 40, 41, 44]
id: 229 run_id: 38 name: run000038_real000229_1
id: 231 run_id: 39 name: run000039_real000231_1
id: 233 run_id: 40 name: run000040_real000233_1
id: 234 run_id: 41 name: run000041_real000234_1
id: 239 run_id: 44 name: run000044_real000239_1


In [ ]:
from pathlib import Path

# Add/adjust roots where older runs may have been recorded
ROOTS = [
    Path.cwd(),  # current repo
    Path(r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM"),
    Path(r"C:\SWAT\RSWAT"),  # add any other places you used before
]

found_ledgers = []
for root in ROOTS:
    try:
        for p in root.rglob("realizations.jsonl"):
            # avoid duplicates
            if str(p.resolve()) not in {str(x.resolve()) for x in found_ledgers}:
                found_ledgers.append(p)
    except Exception:
        pass

print("Found ledger candidates:")
for i, p in enumerate(found_ledgers, 1):
    print(f"{i:2d}. {p}")

# Quick counts for each candidate
from python_pipeline_scripts.provenance_report import read_ledger
print("\nCounts per ledger:")
for p in found_ledgers:
    try:
        n = len(read_ledger(p))
        print(f"{p} -> {n} records")
    except Exception as e:
        print(f"{p} -> error: {e}")


Found ledger candidates:
 1. C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\provenance\realizations.jsonl

Counts per ledger:
C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\Python_Pipeline_SWAT_Pascal\swat_pipeline\trabajoFM\config\provenance\realizations.jsonl -> 162 records


In [ ]:
from python_pipeline_scripts.provenance_report import realization_report as _rr, summarize_run as _sr, _ledger_path_default, read_ledger


DEFAULT_LEDGER = _ledger_path_default()

def realization_report_from(ledger_path, realization_id):
    return _rr(realization_id, ledger_path=ledger_path)

def summarize_run_from(ledger_path, run_id):
    return _sr(run_id, ledger_path=ledger_path)


realization_report_from(DEFAULT_LEDGER, realization_id)

In [ ]:
print(realization_report(10))

Realization id=10 run_id=None name=mc_000010_2
Created: 2025-09-01T16:11:55.707609+00:00
Engine: module=python_pipeline_scripts.mc_engine version=0.1.0 seed=0 N=4
Base TxtInOut: C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1
Realization folder: C:\SWAT\RSWAT\cubillas\mc_realizations\000010_mc_000010_2

Steps:
  - transform_compute_base_soil_vars | args={'id_col': 'HRU_GIS', 'n_col': 'mean_Nitrogeno_total_porcent_resample_Rediam', 'p_col': 'mean_Fosforo_mg_100g_P205_rediam'} | started=2025-09-01T16:11:55.711138+00:00 ended=2025-09-01T16:11:55.715136+00:00
  - transform_perturb_relative | args={'targets': [{'name': 'N_total_mg_kg', 'mode': 'relative', 'bound': 0.2, 'delta': -0.2}, {'name': 'P_element_mg_kg', 'mode': 'relative', 'bound': 0.2, 'delta': 0.2}]} | started=2025-09-01T16:11:55.715136+00:00 ended=2025-09-01T16:11:55.720171+00:00
  - transform_split_fixed_ratios | args={'splits': [{'src': 'N_total_mg_kg', 'outputs': [{'name': 'Soil NO3 [mg/kg]', 'r

## testing monte Carlo setup with extrem bounds

In [ ]:
for dn, dp in product([-alpha_n, +alpha_n], [-alpha_p, +alpha_p]):
    print(dn, dp)

-0.2 -0.2
-0.2 0.2
0.2 -0.2
0.2 0.2


In [ ]:
print(per_realization_params)

[[{}, {'delta': -0.2}, {'delta': -0.2}, {}, {}, {}], [{}, {'delta': -0.2}, {'delta': 0.2}, {}, {}, {}], [{}, {'delta': 0.2}, {'delta': -0.2}, {}, {}, {}], [{}, {'delta': 0.2}, {'delta': 0.2}, {}, {}, {}]]


## Archive


In [ ]:
#!/usr/bin/env python
from __future__ import annotations

from pathlib import Path
from itertools import product

import geopandas as gpd

from python_pipeline_scripts.raster_agg import raster_zonal_aggregation_to_gpkg
from python_pipeline_scripts.transforms.soil_chm import (
    read_n_p_means_from_csv_to_df,
    mc_transform_write_chm,
)
from python_pipeline_scripts.mc_engine import run_monte_carlo
from python_pipeline_scripts import utils



In [ ]:


def _resolve(p: str, base: Path) -> Path:
    pp = Path(p)
    return pp if pp.is_absolute() else (base / pp).resolve()



# Base dir = used to resolve relative paths
script_dir = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\scripts\script DIFFUSE loads - input .chm"

# 1) Zonal aggregation → GPKG and CSV
output_gpkg = r"..\..\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.gpkg"
raster_folder = r"..\..\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster"
zones_fp = r"C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Archivos de Cesar Ruben Fernandez De Villaran San Juan - swat_cubillas\cubillas_hru\Watershed\Shapes\hru1.shp"

output_gpkg_p = _resolve(output_gpkg, script_dir)
raster_folder_p = _resolve(raster_folder, script_dir)
zones_fp_p = Path(zones_fp)

_ = raster_zonal_aggregation_to_gpkg(
    raster_folder=raster_folder_p,
    zones_fp=zones_fp_p,
    zone_field="HRU_GIS",
    label_field="OBJECTID",
    output_gpkg=output_gpkg_p,
    files_end_with="_rediam.tif",
    stat_operation="mean",
    raster_alias="full_name",
    zone_meaning="HRU",
    overwrite_cache=False,
)

layer_name = "values_by_hru"
gdf = gpd.read_file(output_gpkg_p, layer=layer_name)
csv_output = str(output_gpkg_p).replace(".gpkg", ".csv")
gdf.drop(columns="geometry").to_csv(csv_output, sep=";", index=False)
print(f"Exported to CSV: {csv_output}")

# 2) Monte Carlo — extreme-bound combinations (±20% for N and P)
base_txtinout = r"C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1"
realizations_root = r"C:\SWAT\RSWAT\cubillas\mc_realizations"
results_root = r"C:\SWAT\RSWAT\cubillas\mc_results"

base_txtinout_p = Path(base_txtinout)
realizations_root_p = Path(realizations_root)
results_root_p = Path(results_root)

# Build 4 param sets for corners
alpha_n = 0.20
alpha_p = 0.20
per_params = []
for dn, dp in product([-alpha_n, +alpha_n], [-alpha_p, +alpha_p]):
    per_params.append(
        {
            "bounds": {"N_total_pct": alpha_n, "P2O5_mg100g": alpha_p},
            "deltas": {"N_total_pct": dn, "P2O5_mg100g": dp},
            "id_col": "HRU_GIS",
            "n_col": "mean_Nitrogeno_total_porcent_resample_Rediam",
            "p_col": "mean_Fosforo_mg_100g_P205_rediam",
            # Optional: "pperco_val": 15,
        }
    )


2025-09-01 17:22:24,161 | INFO | python_pipeline_scripts.raster_agg | Zonal aggregation start | zones=C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Archivos de Cesar Ruben Fernandez De Villaran San Juan - swat_cubillas\cubillas_hru\Watershed\Shapes\hru1.shp | rasters_dir=C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster


2025-09-01 17:22:25,242 | INFO | python_pipeline_scripts.raster_agg | Valid HRU features: 878 | bounds=[ 438912.5        4123462.50012207  470812.5        4158362.50012207]
2025-09-01 17:22:25,245 | INFO | python_pipeline_scripts.raster_agg | Found 4 raster(s) matching '_rediam.tif'
2025-09-01 17:22:25,250 | INFO | python_pipeline_scripts.raster_agg | Reprojected -> C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster\temp_cache\Carobo_porcent_Rediam_reproj.tif
2025-09-01 17:22:25,262 | INFO | python_pipeline_scripts.raster_agg | Clipped -> C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcGIS _ base de suelo quimico _ ruben _ 30-05-25\raster\temp_cache\Carobo_porcent_Rediam_clip.tif
2025-09-01 17:23:33,201 | INFO | python_pipeline_scripts.raster_agg | Reprojected -> C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil_ArcGIS_Ruben\ArcG

Exported to CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.csv


In [ ]:

cfg = utils.load_config(Path("../config/config.yaml"))

results = run_monte_carlo(
    N=len(per_params),
    base_txtinout=base_txtinout_p,
    realization_root=realizations_root_p,
    results_root=results_root_p,
    link_file_regexes=[r"^.*\.chm$"],
    outputs_to_copy=["output.std", "*.rch"],
    aggregator=lambda: read_n_p_means_from_csv_to_df(
        csv_output,
        id_col="HRU_GIS",
        n_col="mean_Nitrogeno_total_porcent_resample_Rediam",
        p_col="mean_Fosforo_mg_100g_P205_rediam",
    ),
    transforms=[mc_transform_write_chm],
    exe_path=None,  # relies on config.paths.swat_executable
    seed=0,
    expect_plus=False,
    config=cfg,
    include_base_run=True,
    create_workspace_copy=True,
    force_recreate_workspace=True,
    per_realization_params=per_params,
)

ok = sum(1 for r in results if r.success)
print(f"Completed {len(results)} realizations; {ok} succeeded.")
for r in results:
    print(f"- {r.name}: success={r.success} code={r.returncode} outputs_dir={r.outputs_dir}")
return 0 if ok == len(results) else 1



2025-09-01 17:28:05,137 | INFO | python_pipeline_scripts.mc_engine | MC start | N=4 | seed=0
2025-09-01 17:28:05,248 | INFO | python_pipeline_scripts.transforms.soil_chm | Loaded HRU CSV: C:\Users\Usuario\OneDrive - UNIVERSIDAD DE HUELVA\Granada\TrabajoFM\Genil GEO_INFO_POOL\Input Data\Diffuse loads\Soil chemical composition\python calculated hru stats\hru_chem_stats.csv | rows=878


2025-09-01 17:28:06,338 | INFO | python_pipeline_scripts.transforms.soil_chm | Wrote 0 CHM files to C:\SWAT\RSWAT\cubillas\mc_realizations\000005_mc_000005_1
2025-09-01 17:28:06,952 | INFO | python_pipeline_scripts.transforms.soil_chm | Wrote 0 CHM files to C:\SWAT\RSWAT\cubillas\mc_realizations\000006_mc_000006_2
2025-09-01 17:28:07,403 | INFO | python_pipeline_scripts.transforms.soil_chm | Wrote 0 CHM files to C:\SWAT\RSWAT\cubillas\mc_realizations\000007_mc_000007_3
2025-09-01 17:28:07,859 | INFO | python_pipeline_scripts.transforms.soil_chm | Wrote 0 CHM files to C:\SWAT\RSWAT\cubillas\mc_realizations\000008_mc_000008_4
2025-09-01 17:28:07,870 | INFO | python_pipeline_scripts.realizations | Removing existing workspace to recreate: C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BASE_set-219\TxtInOut_1_work\TxtInOut
2025-09-01 17:28:24,618 | INFO | python_pipeline_scripts.realizations | Creating full workspace copy at C:\SWAT\RSWAT\cubillas\cubillas_set_219_ruben\cubillas_BAS

Completed 5 realizations; 0 succeeded.
- base_cubillas_BASE_set-219: success=False code=127 outputs_dir=C:\SWAT\RSWAT\cubillas\mc_results\base_cubillas_BASE_set-219
- mc_000005_1: success=False code=127 outputs_dir=C:\SWAT\RSWAT\cubillas\mc_results\mc_000005_1
- mc_000006_2: success=False code=127 outputs_dir=C:\SWAT\RSWAT\cubillas\mc_results\mc_000006_2
- mc_000007_3: success=False code=127 outputs_dir=C:\SWAT\RSWAT\cubillas\mc_results\mc_000007_3
- mc_000008_4: success=False code=127 outputs_dir=C:\SWAT\RSWAT\cubillas\mc_results\mc_000008_4


SyntaxError: 'return' outside function (936677042.py, line 31)